# Pooled SpherIDiff with Noise Conditioning and Coefficient Filters

This notebook trains one variant of the pooled SpherIDiff with 3 levels and dense spectral weights (`spheridiff_pooled3_dense`). It combines two changes that each improved the pooled model in the earlier experiment with targeted changes, and that act on different parts of the model.

- **Noise conditioning.** The network receives the noise level of every frame. A small embedding of the level is added to the features after the uplift, and it starts at zero. The noise level is known in training, where it is sampled, and in sampling, where it follows the DDIM schedule and is zero for the clean context frame. Giving it to the network is a departure from WIND, which infers it from the data, and it leaves the training objective unchanged.
- **Coefficient filters.** Each channel c gets its own filter over the harmonic coefficients, kappa_c(l, m), added to the shared spectral weight in every block, so that W_c = W + diag(kappa_c(l, m)). It starts at zero. In the earlier experiment the filter depended on the degree l only. Depending also on the order m lets each channel weight zonal and sectoral patterns of the same degree differently, while the filter stays symmetric under rotations about the polar axis.

| Variant | Changes | Parameters |
|---|---|---|
| `pooled3_noise_coeff` | noise conditioning and coefficient filters | 2,925,310 |

The variant is pre-trained on the global ERA5 data at 5.6 degrees for 25,000 steps (`pilot_25k`) and evaluated under the fixed protocol `final_v1` on the test year 2021. It is then fine-tuned for 5,000 steps (`finetune`) on the regional patch at 1.5 degrees over South Asia, with the same Slepian levels as before (L' = 120, 60, 30), and evaluated on that patch zero-shot and after fine-tuning. On the patch, the restricted weight is one frozen M x M matrix per channel, and the trainable update stays shared over the channels.

The earlier runs serve as references: the unchanged model from the pooled experiment, and the variants with degree filters and with noise conditioning from the experiment with targeted changes.

The notebook is designed for a background run on two T4 GPUs. The single variant runs on the first GPU, and the second GPU stays free. It reads the global data cache from the dataset `thesyeddaniyal/era5-pilot-6degrees` and the regional data cache from the dataset `thesyeddaniyal/era5-1-5-southasia`. The pre-trained weights are created in this session. The results are packed into `pooled_noise_coeff_results.zip` in the working folder.

In [ ]:
import os
import sys
from pathlib import Path

import torch

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUNS_ROOT = WORK_ROOT / "runs"
FINAL_PROTOCOL = "final_v1"
ZERO_SHOT_SUFFIX = "_zeroshot"
FIRST_GPU_VARIANTS = ["pooled3_noise_coeff"]
SECOND_GPU_VARIANTS = []
VARIANTS = FIRST_GPU_VARIANTS + SECOND_GPU_VARIANTS
PRETRAINING = ["pilot_" + name for name in VARIANTS]
FINETUNING = ["finetune_" + name for name in VARIANTS]
ZERO_SHOT = [name + ZERO_SHOT_SUFFIX for name in FINETUNING]
ARCHIVE = WORK_ROOT / "pooled_noise_coeff_results.zip"

## Inputs and hardware

The repository code is cloned from GitHub (`https://github.com/syeddaniyalg/spheri-diff`) into the working folder, which requires internet access for the notebook. Within each attached dataset, the data cache is the folder that holds `stats.npz`. When the expected input path of a dataset does not exist, the whole of `/kaggle/input` is searched for a folder of the expected name.

In [ ]:
import subprocess

REPOSITORY_URL = "https://github.com/syeddaniyalg/spheri-diff"
CODE_ROOT = WORK_ROOT / "spheri-diff"
GLOBAL_INPUT = INPUT_ROOT / "datasets" / "thesyeddaniyal" / "era5-pilot-6degrees"
REGIONAL_INPUT = INPUT_ROOT / "datasets" / "thesyeddaniyal" / "era5-1-5-southasia"


def locate_cache(expected, folder_name):
    roots = [expected] if expected.exists() else [INPUT_ROOT]
    for root in roots:
        for current, _, files in os.walk(root):
            if "stats.npz" in files and (expected.exists() or Path(current).name == folder_name):
                return Path(current)
    raise FileNotFoundError(f"no data cache named {folder_name} was found under {roots[0]}")


if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)
GLOBAL_CACHE = locate_cache(GLOBAL_INPUT, "era5_5p6deg")
REGIONAL_CACHE = locate_cache(REGIONAL_INPUT, "era5_1p5deg_south_asia")
sys.path.insert(0, str(CODE_ROOT / "src"))
print("code:", CODE_ROOT)
print("global data:", GLOBAL_CACHE)
print("regional data:", REGIONAL_CACHE)
for index in range(torch.cuda.device_count()):
    print("gpu", index, torch.cuda.get_device_name(index))

## Running the jobs

The variant runs as one queue of separate processes on the first GPU. The queue runs five jobs in order: pre-training, the global final evaluation, fine-tuning, the regional final evaluation of the zero-shot model and the regional final evaluation of the fine-tuned model. The second GPU is not used.

Every job prints its lines into the notebook, prefixed with the elapsed time, the GPU and the job name, and the same lines are kept in `runs/_logs/parallel.log`. The jobs and the packing of the results share one cell with a `try` and a `finally`, so the figures, the status file, the summary tables and the archive are produced even when a job fails or the cell is interrupted. The archive is also refreshed every 30 minutes.

In [ ]:
from spheridiff.analysis.final_table import write_final_summary
from spheridiff.kaggle.finalise import finalise, guarded, refresh_archive
from spheridiff.kaggle.jobs import final_job, finetune_job, place_queues, train_job
from spheridiff.kaggle.runner import ParallelRunner, RunContext

GPUS = torch.cuda.device_count()
CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, RUNS_ROOT / "_logs", GLOBAL_CACHE)
REGIONAL_SUMMARY = RUNS_ROOT / "regional"


def variant_jobs(name):
    pretraining, finetuning = "pilot_" + name, "finetune_" + name
    return [
        train_job(pretraining, GLOBAL_CACHE),
        final_job(pretraining, FINAL_PROTOCOL, cache_root=GLOBAL_CACHE),
        finetune_job(finetuning, RUNS_ROOT, REGIONAL_CACHE),
        final_job(finetuning, FINAL_PROTOCOL, finetuning + ZERO_SHOT_SUFFIX, REGIONAL_CACHE),
        final_job(finetuning, FINAL_PROTOCOL, cache_root=REGIONAL_CACHE),
    ]


def queue(variants):
    return [job for name in variants for job in variant_jobs(name)]


def regional_summary():
    REGIONAL_SUMMARY.mkdir(parents=True, exist_ok=True)
    write_final_summary(REGIONAL_SUMMARY, [(RUNS_ROOT, ZERO_SHOT + FINETUNING)])


PLACEMENT = [(0, queue(FIRST_GPU_VARIANTS))]


def refresh():
    return refresh_archive(RUNS_ROOT, ARCHIVE)


RUNNER = None
try:
    RUNNER = ParallelRunner(place_queues(PLACEMENT, GPUS), CONTEXT, refresh=refresh)
    RUNNER.run()
finally:
    OUTCOMES = RUNNER.outcomes() if RUNNER else []
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    guarded(RUNS_ROOT, "regional_summary", regional_summary)
    finalise(RUNS_ROOT, ARCHIVE, OUTCOMES, PRETRAINING, final_names=PRETRAINING)
print(OUTCOMES)
print(ARCHIVE, f"{ARCHIVE.stat().st_size / 1024**2:.1f} MB")

## Results

The pre-training summaries list the steps, training time, time per step, peak memory and final loss of every variant, and the figures show the validation error of the variant during pre-training. The global table (`final_evaluation_summary.txt`) lists the pre-trained variant under the fixed protocol on the global test year. The regional table (`regional/final_evaluation_summary.txt`) lists every variant on the patch, zero-shot (0 trained steps) and after fine-tuning. Both tables give the number of trainable parameters, the trained steps, the forecast error and CRPS, and for each task and method the normalised error on the unobserved entries, with the reference of the task as the last row.

In [ ]:
import json

for name in PRETRAINING + FINETUNING:
    summary = RUNS_ROOT / name / "summary.json"
    print(name, json.loads(summary.read_text()) if summary.exists() else "no summary, see runs/_logs")

for table in [RUNS_ROOT / "final_evaluation_summary.txt", REGIONAL_SUMMARY / "final_evaluation_summary.txt"]:
    if table.exists():
        print(table.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import Image, display

for figure in ["evaluation_curves.png", "training_loss.png"]:
    path = RUNS_ROOT / figure
    if path.exists():
        display(Image(filename=str(path)))